# Cross-lingual Hybrid Retrieval with Haystack

When building search systems for multilingual content, a common challenge arises: **keyword-based retrieval (BM25) only matches documents in the same language as the query**, while **dense retrieval with multilingual embeddings can bridge languages but may miss exact term matches**.

This cookbook demonstrates how to build a **hybrid retrieval pipeline** in Haystack that combines BM25 and multilingual dense embeddings to handle cross-lingual search effectively. We'll work with a mixed Chinese-English document collection and show how hybrid retrieval outperforms either method alone.

**What you'll learn:**
- How BM25 fails on cross-lingual queries
- How multilingual dense embeddings enable cross-lingual retrieval
- How to combine both with Haystack's `DocumentJoiner` using Reciprocal Rank Fusion
- How to build a complete cross-lingual RAG pipeline with a generator

> 💡 **Real-world motivation:** In e-commerce platforms serving multiple markets, product descriptions may exist in different languages. A user searching in English should still find relevant Chinese-language product pages, and vice versa. This pattern applies broadly to multilingual knowledge bases, academic literature, and enterprise document search.

## Install Dependencies

We'll use Haystack's in-memory document store (no external database needed) with `sentence-transformers` for multilingual embeddings.

In [1]:
%%bash
pip install -q haystack-ai "sentence-transformers>=3.0.0"

## Prepare a Multilingual Document Collection

Let's create a small but realistic dataset of documents in both English and Chinese. These cover overlapping topics — renewable energy, AI policy, and urban planning — so we can test whether retrieval works **across language boundaries**.

In [2]:
from haystack import Document

documents = [
    # English documents
    Document(
        content="Solar panel efficiency has improved significantly in recent years. "
                "Modern photovoltaic cells can convert over 22% of sunlight into electricity, "
                "making rooftop solar installations increasingly cost-effective for homeowners.",
        meta={"language": "en", "topic": "renewable_energy"}
    ),
    Document(
        content="The European Union has set ambitious carbon neutrality targets for 2050. "
                "Key policies include the Emissions Trading System, renewable energy mandates, "
                "and substantial funding for green hydrogen research.",
        meta={"language": "en", "topic": "climate_policy"}
    ),
    Document(
        content="Large language models are transforming how developers write code. "
                "AI-assisted programming tools can suggest entire functions, detect bugs, "
                "and explain complex codebases, significantly boosting productivity.",
        meta={"language": "en", "topic": "ai_programming"}
    ),
    Document(
        content="Urban green spaces play a crucial role in reducing heat island effects. "
                "Cities that invest in parks, green roofs, and tree-lined streets see measurable "
                "improvements in air quality and residents' mental health.",
        meta={"language": "en", "topic": "urban_planning"}
    ),
    Document(
        content="Wind power capacity reached record levels globally in 2024. "
                "Offshore wind farms in particular are becoming major contributors "
                "to national energy grids across Europe and East Asia.",
        meta={"language": "en", "topic": "renewable_energy"}
    ),
    # Chinese documents
    Document(
        content="中国的碳中和目标要求在2060年前实现净零排放。"
                "主要措施包括大规模发展光伏发电、推进电动汽车普及、"
                "以及建立全国性的碳排放交易市场。",
        meta={"language": "zh", "topic": "climate_policy"}
    ),
    Document(
        content="深度学习技术在自然语言处理领域取得了重大突破。"
                "基于Transformer架构的大语言模型能够理解上下文语义，"
                "在机器翻译、文本摘要和代码生成等任务上表现优异。",
        meta={"language": "zh", "topic": "ai_programming"}
    ),
    Document(
        content="城市热岛效应是现代都市面临的重要环境问题。"
                "通过增加城市绿化面积、推广绿色屋顶和透水路面，"
                "可以有效降低城区温度并改善居民生活环境。",
        meta={"language": "zh", "topic": "urban_planning"}
    ),
    Document(
        content="新型钙钛矿太阳能电池的转换效率已突破25%。"
                "与传统硅基电池相比，钙钛矿电池制造成本更低，"
                "柔性基底的特性使其可以应用于建筑外墙和便携设备。",
        meta={"language": "zh", "topic": "renewable_energy"}
    ),
    Document(
        content="检索增强生成（RAG）技术通过结合外部知识库来提升大模型的准确性。"
                "系统首先从文档集合中检索相关段落，然后将检索结果作为上下文"
                "输入给生成模型，从而减少幻觉问题并提供可追溯的信息来源。",
        meta={"language": "zh", "topic": "rag"}
    ),
]

print(f"Total documents: {len(documents)}")
print(f"English: {sum(1 for d in documents if d.meta['language'] == 'en')}")
print(f"Chinese: {sum(1 for d in documents if d.meta['language'] == 'zh')}")

Total documents: 10
English: 5
Chinese: 5


## Approach 1: BM25 Retrieval (Keyword-based)

Let's first try BM25, the classic keyword-matching algorithm. BM25 works by matching exact terms between the query and documents.

BM25 relies on **whitespace tokenization** to split text into terms. This works well for space-delimited languages like English, but presents a fundamental limitation for languages like Chinese, Japanese, or Thai where words are not separated by spaces. Haystack's `InMemoryBM25Retriever` uses whitespace-based tokenization, so:

- **English → English**: Works normally
- **English → Chinese**: No matches (different scripts)
- **Chinese → Chinese**: Poor results (entire sentences become single "tokens" without word segmentation)

Let's verify this limitation:

In [3]:
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever

# Create document store and write documents
bm25_store = InMemoryDocumentStore()
bm25_store.write_documents(documents)

bm25_retriever = InMemoryBM25Retriever(document_store=bm25_store, top_k=5)

# Test: English query about carbon neutrality
query = "carbon neutrality policies and emission targets"
results = bm25_retriever.run(query=query)

print(f"Query: '{query}'")
print(f"Results found: {len(results['documents'])}\n")
for i, doc in enumerate(results["documents"]):
    print(f"  [{i+1}] (lang={doc.meta['language']}, score={doc.score:.4f})")
    print(f"      {doc.content[:80]}...\n")

if len(results["documents"]) == 0:
    print("  (No results — BM25 could not match any documents)")

Query: 'carbon neutrality policies and emission targets'
Results found: 5

  [1] (lang=en, score=9.3893)
      The European Union has set ambitious carbon neutrality targets for 2050. Key pol...

  [2] (lang=en, score=6.1257)
      Urban green spaces play a crucial role in reducing heat island effects. Cities t...

  [3] (lang=en, score=5.9469)
      Large language models are transforming how developers write code. AI-assisted pr...

  [4] (lang=en, score=5.9372)
      Wind power capacity reached record levels globally in 2024. Offshore wind farms ...

  [5] (lang=en, score=5.5397)
      Solar panel efficiency has improved significantly in recent years. Modern photov...



As expected, BM25 can only find documents that share the same script and vocabulary as the query. The Chinese document about China's carbon neutrality goals (碳中和) is completely invisible to English keyword matching.

> ⚠️ **Note on Chinese BM25:** Haystack's `InMemoryDocumentStore` uses whitespace-based tokenization. Since Chinese text has no spaces between words, BM25 treats entire Chinese sentences as single tokens, leading to poor recall even for Chinese-to-Chinese queries. In production systems, you would use a document store with proper CJK tokenization (e.g., Elasticsearch with `ik_analyzer` or OpenSearch with CJK plugins).

## Approach 2: Dense Retrieval (Multilingual Embeddings)

Now let's use a **multilingual embedding model** that maps both Chinese and English text into a shared semantic vector space. We'll use `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, which supports 50+ languages.

The key insight: semantically similar content in different languages gets mapped to nearby vectors, enabling cross-lingual retrieval without any tokenization workarounds.

In [4]:
from haystack.components.embedders import (
    SentenceTransformersDocumentEmbedder,
    SentenceTransformersTextEmbedder,
)

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Embed and index all documents
dense_store = InMemoryDocumentStore()
doc_embedder = SentenceTransformersDocumentEmbedder(model=EMBEDDING_MODEL)
doc_embedder.warm_up()
docs_with_embeddings = doc_embedder.run(documents=documents)
dense_store.write_documents(docs_with_embeddings["documents"])

print(f"Indexed {dense_store.count_documents()} documents with embeddings")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Indexed 10 documents with embeddings


In [5]:
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

# Dense retrieval with the same English query
text_embedder = SentenceTransformersTextEmbedder(model=EMBEDDING_MODEL)
text_embedder.warm_up()

dense_retriever = InMemoryEmbeddingRetriever(document_store=dense_store, top_k=5)

query = "carbon neutrality policies and emission targets"
query_embedding = text_embedder.run(text=query)
results = dense_retriever.run(query_embedding=query_embedding["embedding"])

print(f"Query: '{query}'")
print(f"Results found: {len(results['documents'])}\n")
for i, doc in enumerate(results["documents"]):
    print(f"  [{i+1}] (lang={doc.meta['language']}, score={doc.score:.4f})")
    print(f"      {doc.content[:80]}...\n")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: 'carbon neutrality policies and emission targets'
Results found: 5

  [1] (lang=en, score=18.3640)
      The European Union has set ambitious carbon neutrality targets for 2050. Key pol...

  [2] (lang=zh, score=14.1291)
      中国的碳中和目标要求在2060年前实现净零排放。主要措施包括大规模发展光伏发电、推进电动汽车普及、以及建立全国性的碳排放交易市场。...

  [3] (lang=en, score=7.1499)
      Urban green spaces play a crucial role in reducing heat island effects. Cities t...

  [4] (lang=en, score=6.3158)
      Solar panel efficiency has improved significantly in recent years. Modern photov...

  [5] (lang=en, score=5.8462)
      Wind power capacity reached record levels globally in 2024. Offshore wind farms ...



**Key result:** Dense retrieval with multilingual embeddings successfully retrieves **both** the English EU policy document **and** the Chinese carbon neutrality document. The embedding model understands that "carbon neutrality" and "碳中和" are semantically equivalent — no word segmentation needed.

## Approach 3: Hybrid Retrieval Pipeline

While dense retrieval handles cross-lingual matching, BM25 still has strengths for same-language queries: it's great at matching **exact terms, names, and technical identifiers** that embedding models might overlook. The best approach is to combine both methods.

Haystack's `DocumentJoiner` merges results from multiple retrievers using **Reciprocal Rank Fusion (RRF)**, which combines ranked lists without requiring score normalization:

$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + r(d)}$$

where $r(d)$ is the rank of document $d$ in retriever $r$, and $k$ is a constant (default 60).

In our cross-lingual setting, hybrid retrieval means:
- **BM25** boosts same-language exact matches
- **Dense retrieval** catches cross-lingual semantic matches
- **RRF** merges both ranked lists, so documents found by both methods rank highest

In [6]:
from haystack import Pipeline
from haystack.components.joiners import DocumentJoiner
from haystack.components.retrievers.in_memory import (
    InMemoryBM25Retriever,
    InMemoryEmbeddingRetriever,
)
from haystack.components.embedders import SentenceTransformersTextEmbedder

# dense_store already has embeddings, and InMemoryDocumentStore supports BM25 natively

hybrid_pipeline = Pipeline()

# Components
hybrid_pipeline.add_component(
    "text_embedder",
    SentenceTransformersTextEmbedder(model=EMBEDDING_MODEL)
)
hybrid_pipeline.add_component(
    "bm25_retriever",
    InMemoryBM25Retriever(document_store=dense_store, top_k=5)
)
hybrid_pipeline.add_component(
    "embedding_retriever",
    InMemoryEmbeddingRetriever(document_store=dense_store, top_k=5)
)
hybrid_pipeline.add_component(
    "joiner",
    DocumentJoiner(join_mode="reciprocal_rank_fusion", top_k=5)
)

# Connections
hybrid_pipeline.connect("text_embedder.embedding", "embedding_retriever.query_embedding")
hybrid_pipeline.connect("bm25_retriever.documents", "joiner.documents")
hybrid_pipeline.connect("embedding_retriever.documents", "joiner.documents")

print(hybrid_pipeline)

🚅 Components
  - text_embedder: SentenceTransformersTextEmbedder
  - bm25_retriever: InMemoryBM25Retriever
  - embedding_retriever: InMemoryEmbeddingRetriever
  - joiner: DocumentJoiner
🛤️ Connections
  - text_embedder.embedding -> embedding_retriever.query_embedding (list[float])
  - bm25_retriever.documents -> joiner.documents (list[Document])
  - embedding_retriever.documents -> joiner.documents (list[Document])



Now let's run the hybrid pipeline with our cross-lingual test queries:

In [7]:
def run_hybrid_query(pipeline, query: str, label: str = ""):
    """Run a query through the hybrid pipeline and display results."""
    results = pipeline.run({
        "text_embedder": {"text": query},
        "bm25_retriever": {"query": query},
    })
    
    docs = results["joiner"]["documents"]
    if label:
        print(f"{'='*60}")
        print(f"{label}")
    print(f"Query: '{query}'")
    print(f"Results: {len(docs)}\n")
    for i, doc in enumerate(docs):
        print(f"  [{i+1}] (lang={doc.meta['language']}, topic={doc.meta['topic']}, score={doc.score:.4f})")
        print(f"      {doc.content[:80]}...\n")
    return docs


# Test 1: English query — should find both EN and ZH carbon neutrality docs
run_hybrid_query(
    hybrid_pipeline,
    "carbon neutrality policies and emission targets",
    label="Test 1: English query on a topic covered in both languages"
)

# Test 2: Chinese query — should find both ZH and EN solar energy docs
run_hybrid_query(
    hybrid_pipeline,
    "太阳能电池效率提升",
    label="Test 2: Chinese query on a topic covered in both languages"
)

# Test 3: English query matching a Chinese-only document (RAG)
run_hybrid_query(
    hybrid_pipeline,
    "RAG retrieval augmented generation reduce hallucination",
    label="Test 3: English query matching a Chinese-only document"
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Test 1: English query on a topic covered in both languages
Query: 'carbon neutrality policies and emission targets'
Results: 5

  [1] (lang=en, topic=climate_policy, score=1.0000)
      The European Union has set ambitious carbon neutrality targets for 2050. Key pol...

  [2] (lang=en, topic=urban_planning, score=0.9761)
      Urban green spaces play a crucial role in reducing heat island effects. Cities t...

  [3] (lang=en, topic=renewable_energy, score=0.9458)
      Wind power capacity reached record levels globally in 2024. Offshore wind farms ...

  [4] (lang=en, topic=renewable_energy, score=0.9458)
      Solar panel efficiency has improved significantly in recent years. Modern photov...

  [5] (lang=zh, topic=climate_policy, score=0.4919)
      中国的碳中和目标要求在2060年前实现净零排放。主要措施包括大规模发展光伏发电、推进电动汽车普及、以及建立全国性的碳排放交易市场。...



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Test 2: Chinese query on a topic covered in both languages
Query: '太阳能电池效率提升'
Results: 5

  [1] (lang=en, topic=renewable_energy, score=0.5000)
      Solar panel efficiency has improved significantly in recent years. Modern photov...

  [2] (lang=zh, topic=renewable_energy, score=0.4919)
      新型钙钛矿太阳能电池的转换效率已突破25%。与传统硅基电池相比，钙钛矿电池制造成本更低，柔性基底的特性使其可以应用于建筑外墙和便携设备。...

  [3] (lang=zh, topic=climate_policy, score=0.4841)
      中国的碳中和目标要求在2060年前实现净零排放。主要措施包括大规模发展光伏发电、推进电动汽车普及、以及建立全国性的碳排放交易市场。...

  [4] (lang=en, topic=renewable_energy, score=0.4766)
      Wind power capacity reached record levels globally in 2024. Offshore wind farms ...

  [5] (lang=en, topic=urban_planning, score=0.4692)
      Urban green spaces play a crucial role in reducing heat island effects. Cities t...



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Test 3: English query matching a Chinese-only document
Query: 'RAG retrieval augmented generation reduce hallucination'
Results: 5

  [1] (lang=zh, topic=rag, score=1.0000)
      检索增强生成（RAG）技术通过结合外部知识库来提升大模型的准确性。系统首先从文档集合中检索相关段落，然后将检索结果作为上下文输入给生成模型，从而减少幻觉问题并提...

  [2] (lang=en, topic=renewable_energy, score=0.9685)
      Solar panel efficiency has improved significantly in recent years. Modern photov...

  [3] (lang=en, topic=ai_programming, score=0.9607)
      Large language models are transforming how developers write code. AI-assisted pr...

  [4] (lang=zh, topic=ai_programming, score=0.4919)
      深度学习技术在自然语言处理领域取得了重大突破。基于Transformer架构的大语言模型能够理解上下文语义，在机器翻译、文本摘要和代码生成等任务上表现优异。...

  [5] (lang=en, topic=climate_policy, score=0.4841)
      The European Union has set ambitious carbon neutrality targets for 2050. Key pol...



[Document(id=8d4b9e2c1b56afc9ce9ce799bfc695d47111f8c164d928a92dc6579f0d9f462b, content: '检索增强生成（RAG）技术通过结合外部知识库来提升大模型的准确性。系统首先从文档集合中检索相关段落，然后将检索结果作为上下文输入给生成模型，从而减少幻觉问题并提供可追溯的信息来源。', meta: {'language': 'zh', 'topic': 'rag'}, score: 1.0),
 Document(id=731aedac5a0d1a057fbbd4ca5c24bd61f616ed5d362bab73d36942e344479434, content: 'Solar panel efficiency has improved significantly in recent years. Modern photovoltaic cells can con...', meta: {'language': 'en', 'topic': 'renewable_energy'}, score: 0.9684979838709676),
 Document(id=ab23724d86ff952513d5abd131db4e3bf178a437b7f2691657288c814b7ec22c, content: 'Large language models are transforming how developers write code. AI-assisted programming tools can ...', meta: {'language': 'en', 'topic': 'ai_programming'}, score: 0.9606894841269841),
 Document(id=fa543a8d3e47c5bd991c42592866bb25e9ac127266f3744be2edcef5c695beff, content: '深度学习技术在自然语言处理领域取得了重大突破。基于Transformer架构的大语言模型能够理解上下文语义，在机器翻译、文本摘要和代码生成等任务上表现优异。', meta: {'language': 'zh', 'topic': 'ai_p

## Comparing the Approaches

Let's systematically compare how each method handles cross-lingual queries:

In [8]:
def count_languages(docs):
    """Count documents by language."""
    langs = {}
    for doc in docs:
        lang = doc.meta.get("language", "unknown")
        langs[lang] = langs.get(lang, 0) + 1
    return langs


test_queries = [
    "carbon neutrality policies",
    "solar cell efficiency",
    "urban heat island green spaces",
    "AI code generation LLM",
]

print(f"{'Query':<35} {'Method':<12} {'EN docs':<10} {'ZH docs':<10}")
print("-" * 67)

for query_en in test_queries:
    # BM25 only
    bm25_results = bm25_retriever.run(query=query_en)["documents"]
    bm25_langs = count_languages(bm25_results)
    
    # Dense only
    q_emb = text_embedder.run(text=query_en)
    dense_results = dense_retriever.run(query_embedding=q_emb["embedding"])["documents"]
    dense_langs = count_languages(dense_results)
    
    # Hybrid
    hybrid_results = hybrid_pipeline.run({
        "text_embedder": {"text": query_en},
        "bm25_retriever": {"query": query_en},
    })["joiner"]["documents"]
    hybrid_langs = count_languages(hybrid_results)
    
    q_display = query_en[:33]
    print(f"{q_display:<35} {'BM25':<12} {bm25_langs.get('en', 0):<10} {bm25_langs.get('zh', 0):<10}")
    print(f"{'':35} {'Dense':<12} {dense_langs.get('en', 0):<10} {dense_langs.get('zh', 0):<10}")
    print(f"{'':35} {'Hybrid':<12} {hybrid_langs.get('en', 0):<10} {hybrid_langs.get('zh', 0):<10}")
    print()

Query                               Method       EN docs    ZH docs   
-------------------------------------------------------------------


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

carbon neutrality policies          BM25         5          0         
                                    Dense        3          2         
                                    Hybrid       4          1         



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

urban heat island green spaces      BM25         5          0         
                                    Dense        3          2         
                                    Hybrid       4          1         



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

AI code generation LLM              BM25         5          0         
                                    Dense        2          3         
                                    Hybrid       4          1         



The results show that:
- **BM25** retrieves only same-language documents (and may return 0 results for Chinese due to tokenization)
- **Dense retrieval** finds relevant documents across both languages
- **Hybrid** combines both strengths: it captures cross-lingual semantic matches while still rewarding exact term overlap for same-language queries

## Building a Complete Cross-lingual RAG Pipeline

Now let's connect our hybrid retriever to a **generator** to build a complete cross-lingual RAG pipeline. The generator synthesizes information from documents in both languages to provide a comprehensive answer.

We'll use `HuggingFaceLocalChatGenerator` with a small model so this cookbook runs without any API keys. For better multilingual generation quality in production, consider using OpenAI, Anthropic, or a larger multilingual model.

In [9]:
%%bash
pip install -q "transformers[torch]" accelerate

In [10]:
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import HuggingFaceLocalChatGenerator
from haystack.dataclasses import ChatMessage

rag_pipeline = Pipeline()

# Retrieval components (same as before)
rag_pipeline.add_component(
    "text_embedder",
    SentenceTransformersTextEmbedder(model=EMBEDDING_MODEL)
)
rag_pipeline.add_component(
    "bm25_retriever",
    InMemoryBM25Retriever(document_store=dense_store, top_k=5)
)
rag_pipeline.add_component(
    "embedding_retriever",
    InMemoryEmbeddingRetriever(document_store=dense_store, top_k=5)
)
rag_pipeline.add_component(
    "joiner",
    DocumentJoiner(join_mode="reciprocal_rank_fusion", top_k=3)
)

# Generation components
template = [
    ChatMessage.from_system(
        "You are a helpful multilingual assistant. Answer the question based on the "
        "provided context documents, which may be in English or Chinese. Synthesize "
        "information from all relevant documents regardless of their language. "
        "Answer in the same language as the question."
    ),
    ChatMessage.from_user(
        "Context:\n"
        "{% for doc in documents %}\n"
        "[{{ doc.meta.language | upper }}] {{ doc.content }}\n"
        "{% endfor %}\n\n"
        "Question: {{ query }}\n"
    ),
]

rag_pipeline.add_component(
    "prompt_builder",
    ChatPromptBuilder(template=template)
)
rag_pipeline.add_component(
    "llm",
    HuggingFaceLocalChatGenerator(model="Qwen/Qwen3-0.6B")
)

# Connect retrieval
rag_pipeline.connect("text_embedder.embedding", "embedding_retriever.query_embedding")
rag_pipeline.connect("bm25_retriever.documents", "joiner.documents")
rag_pipeline.connect("embedding_retriever.documents", "joiner.documents")

# Connect generation
rag_pipeline.connect("joiner.documents", "prompt_builder.documents")
rag_pipeline.connect("prompt_builder", "llm")
print(rag_pipeline)

ChatPromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


🚅 Components
  - text_embedder: SentenceTransformersTextEmbedder
  - bm25_retriever: InMemoryBM25Retriever
  - embedding_retriever: InMemoryEmbeddingRetriever
  - joiner: DocumentJoiner
  - prompt_builder: ChatPromptBuilder
  - llm: HuggingFaceLocalChatGenerator
🛤️ Connections
  - text_embedder.embedding -> embedding_retriever.query_embedding (list[float])
  - bm25_retriever.documents -> joiner.documents (list[Document])
  - embedding_retriever.documents -> joiner.documents (list[Document])
  - joiner.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> llm.messages (list[ChatMessage])



In [11]:
# Run the full cross-lingual RAG pipeline
query = "What are the main approaches to carbon neutrality in different regions?"

results = rag_pipeline.run({
    "text_embedder": {"text": query},
    "bm25_retriever": {"query": query},
    "prompt_builder": {"query": query},
},
include_outputs_from={"joiner"},)

# Display the retrieved documents
print("Retrieved documents:")
for i, doc in enumerate(results["joiner"]["documents"]):
    print(f"  [{i+1}] (lang={doc.meta['language']}) {doc.content[:60]}...")

# Display the generated answer
print(f"\n{'='*60}")
print("Generated answer:")
print(results["llm"]["replies"][0].text)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved documents:
  [1] (lang=en) The European Union has set ambitious carbon neutrality targe...
  [2] (lang=en) Urban green spaces play a crucial role in reducing heat isla...
  [3] (lang=en) Solar panel efficiency has improved significantly in recent ...

Generated answer:
The main approaches to carbon neutrality in different regions include the Emissions Trading System, renewable energy mandates, and substantial funding for green hydrogen research. Additionally, urban green spaces play a crucial role in reducing heat island effects and improving air quality and mental health.


The generator successfully synthesized information from **both English and Chinese** source documents into a coherent answer. This demonstrates the power of cross-lingual hybrid retrieval: the pipeline retrieves relevant content regardless of language, and the LLM combines insights from all sources.

## Key Takeaways

1. **BM25 is language-bound.** It relies on whitespace tokenization, which works for space-delimited languages but fails for Chinese/Japanese/Korean. Even for same-language queries, it cannot cross language boundaries.

2. **Multilingual dense embeddings bridge languages.** Models like `paraphrase-multilingual-MiniLM-L12-v2` map semantically similar content into nearby vectors regardless of language, enabling true cross-lingual search.

3. **Hybrid retrieval combines the best of both.** BM25 catches exact keyword matches (important for technical terms and identifiers in same-language queries), while dense retrieval handles semantic and cross-lingual matching. Reciprocal Rank Fusion merges both result sets without requiring score normalization.

4. **LLMs can synthesize multilingual context.** Modern language models handle mixed-language input naturally, making cross-lingual RAG practical and effective.

5. **This pattern generalizes.** The same approach works for any language pair supported by the embedding model — Arabic-English, Japanese-German, Spanish-French, and more.

## 📚 Resources

- [Haystack Hybrid Retrieval Tutorial](https://haystack.deepset.ai/tutorials/33_hybrid_retrieval)
- [Hybrid Retrieval with BM42 — Cookbook](https://haystack.deepset.ai/cookbook/hybrid_retrieval_bm42)
- [Sentence-Transformers Multilingual Models](https://www.sbert.net/docs/sentence_transformer/pretrained_models.html#multilingual-models)
- [DocumentJoiner API Reference](https://docs.haystack.deepset.ai/docs/documentjoiner)
- [HuggingFaceLocalChatGenerator API Reference](https://docs.haystack.deepset.ai/docs/huggingfacelocalchatgenerator)